# BraTS 2021 Kesifsel Veri Analizi (EDA)

Bu notebook, BraTS 2021 Task 1 verisinin proje icin dogru indirildigini ve modelleme oncesi temel ozelliklerini inceler.

Incelenen basliklar:

- Vaka sayisi ve dosya organizasyonu
- Modalite bazli dosya boyutu istatistikleri
- Goruntu boyutlari ve voxel spacing dagilimi
- Modalite bazli yogunluk istatistikleri
- Segmentasyon sinif dagilimi ve tumor hacimleri
- WT / TC / ET bolge tanimlari
- Ornek cok modlu goruntuleme ve segmentasyon overlay'leri

Not: Yogunluk ve segmentasyon analizleri varsayilan olarak deterministik bir orneklem uzerinden calisir. Tum veri setinde calistirmak icin asagidaki `INTENSITY_SAMPLE_CASES` ve `SEGMENTATION_SAMPLE_CASES` degerlerini `None` yapabilirsin.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 11

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "BraTS2021"
MODALITIES = ["t1", "t1ce", "t2", "flair", "seg"]
IMAGE_MODALITIES = ["t1", "t1ce", "t2", "flair"]
EXPECTED_LABELS = {0, 1, 2, 4}
EXPECTED_SHAPE = (240, 240, 155)
EXPECTED_CASE_COUNT = 1251

# None yaparsan ilgili analiz tum vakalarda calisir. Varsayilanlar notebook'u hizli tutar.
RANDOM_SEED = 42
INTENSITY_SAMPLE_CASES = 64
SEGMENTATION_SAMPLE_CASES = 128
MAX_VOXELS_PER_MODALITY_PER_CASE = 50_000

np.random.seed(RANDOM_SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")

## 1. Veri seti organizasyonu

BraTS 2021 Task 1 icin beklenen yapi her vaka klasorunde 4 MR modalitesi ve 1 segmentasyon maskesidir.

In [ ]:
def discover_cases(data_root: Path) -> list[Path]:
    if not data_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {data_root}")
    cases = sorted(path for path in data_root.iterdir() if path.is_dir())
    if not cases:
        raise FileNotFoundError(f"No case directories found under: {data_root}")
    return cases


def case_file(case_dir: Path, modality: str) -> Path:
    return case_dir / f"{case_dir.name}_{modality}.nii.gz"


cases = discover_cases(DATA_ROOT)
print(f"Case count: {len(cases)}")
print(f"Expected case count: {EXPECTED_CASE_COUNT}")
print("First 5 cases:", [case.name for case in cases[:5]])

missing_or_extra = []
for case in cases:
    expected = {case_file(case, modality).name for modality in MODALITIES}
    actual = {path.name for path in case.iterdir() if path.is_file()}
    missing = sorted(expected - actual)
    extra = sorted(actual - expected)
    if missing or extra:
        missing_or_extra.append({"case_id": case.name, "missing": missing, "extra": extra})

print(f"Cases with missing/extra files: {len(missing_or_extra)}")
if missing_or_extra:
    pd.DataFrame(missing_or_extra).head(20)
else:
    print("All cases have the expected 5 files.")

## 2. Dosya boyutu istatistikleri

Bu tablo, her modalite icin dosya boyutlarini MB cinsinden ozetler.

In [ ]:
manifest_rows = []
for case in cases:
    for modality in MODALITIES:
        path = case_file(case, modality)
        manifest_rows.append(
            {
                "case_id": case.name,
                "modality": modality,
                "path": str(path.relative_to(PROJECT_ROOT)),
                "size_mb": path.stat().st_size / (1024**2),
            }
        )

manifest_df = pd.DataFrame(manifest_rows)
size_stats = manifest_df.groupby("modality")["size_mb"].agg(["count", "min", "mean", "median", "max"]).round(3)
size_stats

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=manifest_df, x="modality", y="size_mb", order=MODALITIES)
plt.title("Modalite bazli dosya boyutu dagilimi")
plt.xlabel("Modalite")
plt.ylabel("Dosya boyutu (MB)")
plt.show()

## 3. Goruntu boyutlari ve voxel spacing

BraTS 2021 verisinin genellikle skull-stripped, co-registered ve 1 mm isotropic spacing ile geldigini dogruluyoruz.

In [ ]:
metadata_rows = []
unreadable = []

for case in cases:
    for modality in MODALITIES:
        path = case_file(case, modality)
        try:
            image = nib.load(str(path))
        except Exception as exc:
            unreadable.append({"case_id": case.name, "modality": modality, "error": str(exc)})
            continue
        metadata_rows.append(
            {
                "case_id": case.name,
                "modality": modality,
                "shape": tuple(image.shape),
                "spacing": tuple(float(v) for v in image.header.get_zooms()[:3]),
                "dtype": str(image.get_data_dtype()),
            }
        )

metadata_df = pd.DataFrame(metadata_rows)
print(f"Unreadable files: {len(unreadable)}")
print(f"Unique shapes: {metadata_df['shape'].nunique()}")
print(f"Unique spacings: {metadata_df['spacing'].nunique()}")
print(f"Expected shape present in all files: {(metadata_df['shape'] == EXPECTED_SHAPE).all()}")
metadata_df.groupby(["shape", "spacing"]).size().reset_index(name="count")

## 4. Yardimci fonksiyonlar

Asagidaki fonksiyonlar yogunluk, segmentasyon ve gorsellestirme analizlerinde kullanilir.

In [ ]:
def select_cases(cases: list[Path], n: int | None) -> list[Path]:
    if n is None or n >= len(cases):
        return list(cases)
    rng = np.random.default_rng(RANDOM_SEED)
    indices = np.sort(rng.choice(len(cases), size=n, replace=False))
    return [cases[i] for i in indices]


def load_volume(path: Path, dtype=np.float32) -> np.ndarray:
    return np.asanyarray(nib.load(str(path)).dataobj).astype(dtype, copy=False)


def nonzero_sample(values: np.ndarray, max_voxels: int) -> np.ndarray:
    values = values[values != 0]
    if values.size <= max_voxels:
        return values
    rng = np.random.default_rng(RANDOM_SEED)
    indices = rng.choice(values.size, size=max_voxels, replace=False)
    return values[indices]


def tumor_regions(seg: np.ndarray) -> dict[str, np.ndarray]:
    return {
        "WT": np.isin(seg, [1, 2, 4]),
        "TC": np.isin(seg, [1, 4]),
        "ET": seg == 4,
    }


def normalize_slice(slice_2d: np.ndarray) -> np.ndarray:
    values = slice_2d.astype(np.float32)
    nonzero = values[values != 0]
    if nonzero.size == 0:
        return values
    lo, hi = np.percentile(nonzero, [1, 99])
    return np.clip((values - lo) / (hi - lo + 1e-8), 0, 1)


def middle_tumor_slices(seg: np.ndarray) -> dict[str, int]:
    tumor = seg > 0
    coords = np.argwhere(tumor)
    if coords.size == 0:
        return {"axial": seg.shape[2] // 2, "coronal": seg.shape[1] // 2, "sagittal": seg.shape[0] // 2}
    center = np.round(coords.mean(axis=0)).astype(int)
    return {"sagittal": int(center[0]), "coronal": int(center[1]), "axial": int(center[2])}


def overlay_segmentation(base: np.ndarray, seg: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    base_norm = normalize_slice(base)
    rgb = np.stack([base_norm, base_norm, base_norm], axis=-1)
    colors = {
        1: np.array([1.0, 0.0, 0.0]),  # NCR/NET
        2: np.array([0.0, 1.0, 0.0]),  # ED
        4: np.array([0.0, 0.0, 1.0]),  # ET
    }
    for label, color in colors.items():
        mask = seg == label
        rgb[mask] = (1 - alpha) * rgb[mask] + alpha * color
    return rgb

## 5. Yogunluk dagilimlari

Her modalite icin non-zero voxel'ler uzerinden yogunluk istatistikleri hesaplanir. Arka plan sifir oldugu icin istatistiklere dahil edilmez.

In [ ]:
intensity_cases = select_cases(cases, INTENSITY_SAMPLE_CASES)
print(f"Intensity analysis cases: {len(intensity_cases)} / {len(cases)}")

intensity_rows = []
hist_samples = []

for case in intensity_cases:
    for modality in IMAGE_MODALITIES:
        volume = load_volume(case_file(case, modality), dtype=np.float32)
        sampled = nonzero_sample(volume, MAX_VOXELS_PER_MODALITY_PER_CASE)
        if sampled.size == 0:
            continue
        intensity_rows.append(
            {
                "case_id": case.name,
                "modality": modality,
                "voxel_count_nonzero": int((volume != 0).sum()),
                "min": float(sampled.min()),
                "p1": float(np.percentile(sampled, 1)),
                "mean": float(sampled.mean()),
                "std": float(sampled.std()),
                "p99": float(np.percentile(sampled, 99)),
                "max": float(sampled.max()),
            }
        )
        if len(hist_samples) < 20:
            hist_samples.append(pd.DataFrame({"modality": modality, "intensity": sampled}))

intensity_df = pd.DataFrame(intensity_rows)
intensity_summary = intensity_df.groupby("modality")[["min", "p1", "mean", "std", "p99", "max"]].mean().round(2)
intensity_summary

In [ ]:
hist_df = pd.concat(hist_samples, ignore_index=True)
g = sns.FacetGrid(hist_df, col="modality", col_wrap=2, sharex=False, sharey=False, height=3.2)
g.map_dataframe(sns.histplot, x="intensity", bins=80, stat="density")
g.set_titles("{col_name}")
g.fig.suptitle("Modalite bazli yogunluk histogramlari (orneklem)", y=1.03)
plt.show()

## 6. Segmentasyon analizi

BraTS etiketleri:

- `0`: Background
- `1`: NCR/NET, nekrotik veya non-enhancing tumor core
- `2`: ED, peritumoral edema
- `4`: ET, enhancing tumor

Bolge tanimlari:

- `WT = 1 + 2 + 4`
- `TC = 1 + 4`
- `ET = 4`

In [ ]:
segmentation_cases = select_cases(cases, SEGMENTATION_SAMPLE_CASES)
print(f"Segmentation analysis cases: {len(segmentation_cases)} / {len(cases)}")

label_rows = []
region_rows = []
invalid_label_cases = []

for case in segmentation_cases:
    seg_path = case_file(case, "seg")
    image = nib.load(str(seg_path))
    seg = np.asanyarray(image.dataobj).astype(np.int16, copy=False)
    labels, counts = np.unique(seg, return_counts=True)
    label_set = {int(label) for label in labels}
    if not label_set.issubset(EXPECTED_LABELS):
        invalid_label_cases.append({"case_id": case.name, "labels": sorted(label_set)})

    voxel_volume_mm3 = abs(np.linalg.det(image.affine[:3, :3]))
    count_by_label = {int(label): int(count) for label, count in zip(labels, counts)}
    total_voxels = int(seg.size)
    tumor_voxels = int(sum(count_by_label.get(label, 0) for label in [1, 2, 4]))
    background_voxels = int(count_by_label.get(0, 0))

    for label in [0, 1, 2, 4]:
        voxels = count_by_label.get(label, 0)
        label_rows.append(
            {
                "case_id": case.name,
                "label": label,
                "voxels": voxels,
                "percent": voxels / total_voxels * 100,
                "volume_cm3": voxels * voxel_volume_mm3 / 1000,
            }
        )

    for region_name, region_mask in tumor_regions(seg).items():
        voxels = int(region_mask.sum())
        region_rows.append(
            {
                "case_id": case.name,
                "region": region_name,
                "voxels": voxels,
                "volume_cm3": voxels * voxel_volume_mm3 / 1000,
                "background_to_tumor_ratio": background_voxels / max(tumor_voxels, 1),
            }
        )

label_df = pd.DataFrame(label_rows)
region_df = pd.DataFrame(region_rows)
print(f"Cases with unexpected labels: {len(invalid_label_cases)}")
label_df.groupby("label")[["voxels", "percent", "volume_cm3"]].agg(["mean", "median", "min", "max"]).round(2)

In [ ]:
region_summary = region_df.groupby("region")[["voxels", "volume_cm3"]].agg(["mean", "median", "min", "max"]).round(2)
imbalance_summary = region_df.drop_duplicates("case_id")["background_to_tumor_ratio"].describe().round(2)
display(region_summary)
print("Background / tumor voxel ratio summary")
display(imbalance_summary.to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=label_df[label_df["label"] != 0], x="label", y="volume_cm3", ax=axes[0])
axes[0].set_title("Etiket bazli tumor hacmi dagilimi")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Hacim (cm3)")

sns.boxplot(data=region_df, x="region", y="volume_cm3", order=["WT", "TC", "ET"], ax=axes[1])
axes[1].set_title("WT / TC / ET hacim dagilimi")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Hacim (cm3)")
plt.tight_layout()
plt.show()

## 7. Gorsel ornekler

Tumor hacmine gore kucuk, orta ve buyuk ornek vakalar secilip goruntulenir. Her vaka icin 4 modalite yan yana ve FLAIR uzerinde segmentasyon overlay'i gosterilir.

In [ ]:
wt_volumes = region_df[region_df["region"] == "WT"].sort_values("volume_cm3")
selected_case_ids = []
if len(wt_volumes) >= 3:
    positions = [0, len(wt_volumes) // 2, len(wt_volumes) - 1]
    selected_case_ids = wt_volumes.iloc[positions]["case_id"].tolist()
else:
    selected_case_ids = [case.name for case in segmentation_cases[:3]]

selected_cases = [DATA_ROOT / case_id for case_id in selected_case_ids]
print("Selected visual cases:", selected_case_ids)

In [ ]:
def plot_modalities_for_case(case: Path) -> None:
    volumes = {modality: load_volume(case_file(case, modality), dtype=np.float32) for modality in IMAGE_MODALITIES}
    seg = load_volume(case_file(case, "seg"), dtype=np.int16)
    slices = middle_tumor_slices(seg)
    z = slices["axial"]

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, modality in zip(axes[:4], IMAGE_MODALITIES):
        ax.imshow(normalize_slice(volumes[modality][:, :, z]).T, cmap="gray", origin="lower")
        ax.set_title(f"{case.name} - {modality} axial z={z}")
        ax.axis("off")

    overlay = overlay_segmentation(volumes["flair"][:, :, z], seg[:, :, z])
    axes[4].imshow(overlay.transpose(1, 0, 2), origin="lower")
    axes[4].set_title("FLAIR + seg overlay")
    axes[4].axis("off")
    plt.tight_layout()
    plt.show()


for case in selected_cases:
    plot_modalities_for_case(case)

In [ ]:
def plot_three_planes(case: Path) -> None:
    flair = load_volume(case_file(case, "flair"), dtype=np.float32)
    seg = load_volume(case_file(case, "seg"), dtype=np.int16)
    slices = middle_tumor_slices(seg)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    x = slices["sagittal"]
    axes[0].imshow(overlay_segmentation(flair[x, :, :], seg[x, :, :]).transpose(1, 0, 2), origin="lower")
    axes[0].set_title(f"Sagittal x={x}")
    axes[0].axis("off")

    y = slices["coronal"]
    axes[1].imshow(overlay_segmentation(flair[:, y, :], seg[:, y, :]).transpose(1, 0, 2), origin="lower")
    axes[1].set_title(f"Coronal y={y}")
    axes[1].axis("off")

    z = slices["axial"]
    axes[2].imshow(overlay_segmentation(flair[:, :, z], seg[:, :, z]).transpose(1, 0, 2), origin="lower")
    axes[2].set_title(f"Axial z={z}")
    axes[2].axis("off")

    fig.suptitle(f"{case.name} - FLAIR + segmentasyon overlay")
    plt.tight_layout()
    plt.show()


for case in selected_cases:
    plot_three_planes(case)

## 8. Bulgularin kisa ozeti

Bu hucreyi notebook calistiktan sonra rapor/sunum icin kullanabilirsin:

- Veri seti `data/raw/BraTS2021` altinda organize edildi.
- Her vaka icin 4 modalite ve 1 segmentasyon maskesi mevcut.
- Tum dosyalar beklenen BraTS boyutunda `(240, 240, 155)` ve 1 mm isotropic spacing ile geliyor.
- Segmentasyon etiketleri BraTS standardi olan `{0, 1, 2, 4}` formatinda.
- Background sinifi tumor siniflarina gore cok baskin oldugu icin class imbalance egitimde dikkate alinmali.
- WT, TC ve ET bolgeleri Dice skorlarinin raporlanmasinda kullanilacak ana klinik/yarismaci bolgelerdir.